# Bronze ingestion — local demonstration

Reads configured CSV sources into **in-memory PySpark DataFrames** and prints an operational ingestion summary. No Delta/Parquet files are written in this Windows environment. The reusable read logic lives in `src/bronze.py`.

In [ ]:
from datetime import datetime
from pathlib import Path
import sys

from pyspark.sql import SparkSession

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.bronze import load_json, load_bronze_source

spark = (
    SparkSession.builder
    .appName("InsuranceLakehouseBronzeIngestion")
    .master("local[*]")
    .getOrCreate()
)

sources = load_json(project_root / "config" / "bronze_sources.json")["sources"]
print(f"Spark version: {spark.version}")
print(f"Configured sources: {len(sources)}")

In [ ]:
bronze_dfs = {}
ingestion_log = []
run_start = datetime.now()
seen_sources = set()

for source in sources:
    if not source.get("active", True):
        continue

    source_name = source["source_name"]
    source_start = datetime.now()
    record_count = 0
    status = "SUCCESS"
    error_message = None

    try:
        if source_name in seen_sources:
            raise ValueError(f"Duplicate source configuration: {source_name}")
        seen_sources.add(source_name)

        df = load_bronze_source(spark, project_root, source)
        record_count = df.count()
        bronze_dfs[source_name] = df
    except Exception as error:
        status = "FAILED"
        error_message = str(error)

    source_end = datetime.now()
    ingestion_log.append({
        "source": source_name,
        "file": source["file_name"],
        "status": status,
        "records": record_count,
        "start_time": source_start,
        "end_time": source_end,
        "duration_seconds": round((source_end - source_start).total_seconds(), 2),
        "error": error_message,
    })

run_end = datetime.now()

In [ ]:
successful_sources = sum(item["status"] == "SUCCESS" for item in ingestion_log)
failed_sources = sum(item["status"] == "FAILED" for item in ingestion_log)
total_records = sum(item["records"] for item in ingestion_log)
run_duration = round((run_end - run_start).total_seconds(), 2)

print("BRONZE INGESTION SUMMARY")
print("-" * 65)
for item in ingestion_log:
    print(
        f"{item['status']:7} | {item['source']:10} | "
        f"records={item['records']:5} | "
        f"duration={item['duration_seconds']}s"
    )
    if item["error"]:
        print(f"  ERROR | {item['source']} | {item['error']}")

print("-" * 65)
print(f"Sources processed : {len(ingestion_log)}")
print(f"Successful        : {successful_sources}")
print(f"Failed            : {failed_sources}")
print(f"Records processed : {total_records}")
print(f"Total duration    : {run_duration}s")

if failed_sources:
    raise RuntimeError(f"Bronze ingestion failed for {failed_sources} source(s)")

In [ ]:
successful_sources = sum(item["status"] == "SUCCESS" for item in ingestion_log)
failed_sources = sum(item["status"] == "FAILED" for item in ingestion_log)
total_records = sum(item["records"] for item in ingestion_log)
run_duration = round((run_end - run_start).total_seconds(), 2)

print("BRONZE INGESTION SUMMARY")
print("-" * 65)
for item in ingestion_log:
    print(
        f"{item['status']:7} | {item['source']:10} | "
        f"records={item['records']:5} | "
        f"duration={item['duration_seconds']}s"
    )
    if item["error"]:
        print(f"  ERROR | {item['source']} | {item['error']}")

print("-" * 65)
print(f"Sources processed : {len(ingestion_log)}")
print(f"Successful        : {successful_sources}")
print(f"Failed            : {failed_sources}")
print(f"Records processed : {total_records}")
print(f"Total duration    : {run_duration}s")

if failed_sources:
    raise RuntimeError(f"Bronze ingestion failed for {failed_sources} source(s)")